# Home Credit Default Risk — Notebook 00  
## Baseline Logistic Regression (Application Data Only)

This notebook builds a **baseline Probability of Default (PD)** model using only application-level data.

It establishes a clear **reference model** before introducing external bureau and behavioural features in later notebooks.


## 1. Purpose of this notebook

This notebook answers a simple question:

> How well can we predict default risk using only application data?

This is important because it:

- provides a **baseline benchmark**
- helps measure the **incremental value of external data later**
- reflects how models are often first built in practice

At the end of this notebook, we will have:

- a trained logistic regression model
- baseline evaluation metrics
- exported outputs for later use


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())


## 2. Load the application data

This baseline notebook uses only `application_train.csv`, which contains the target variable.

- `TARGET = 1` → default
- `TARGET = 0` → non-default


In [ ]:
df = pd.read_csv(DATA_DIR / "application_train.csv")

print("Shape:", df.shape)
df.head()


## 3. Review the target balance

Credit datasets are usually imbalanced, with far fewer defaults than non-defaults.  
That is normal and important to recognise before modelling.


In [ ]:
target_count = df["TARGET"].value_counts().sort_index()
target_ratio = df["TARGET"].value_counts(normalize=True).sort_index()

print("Target count:")
print(target_count)
print()
print("Target ratio:")
print(target_ratio)


## 4. Quick data quality review

Before building features, it is useful to review:

- data types
- non-null counts
- missing values


In [ ]:
summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notnull().sum().values,
    "missing_pct": (df.isnull().mean().values * 100).round(2)
}).sort_values("missing_pct", ascending=False)

summary.head(20)


## 5. Basic feature engineering

For the baseline model, we keep the feature logic simple and explainable.

We create a few intuitive variables that reflect:

- affordability
- repayment burden
- leverage
- borrower stability


In [ ]:
df["DAYS_EMPLOYED"] = pd.to_numeric(df["DAYS_EMPLOYED"], errors="coerce")
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

df["DTI"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
df["INST_TO_INCOME"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
df["LTV_PROXY"] = df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"].replace(0, np.nan)
df["AGE"] = -df["DAYS_BIRTH"] / 365
df["EMPLOYMENT_YEARS"] = -df["DAYS_EMPLOYED"] / 365


## 6. Define the baseline feature set

This baseline set focuses on:

- income
- credit amount
- annuity burden
- leverage proxy
- age and employment stability
- household burden


In [ ]:
features = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "DTI",
    "INST_TO_INCOME",
    "LTV_PROXY",
    "AGE",
    "EMPLOYMENT_YEARS",
    "CNT_CHILDREN"
]

target = "TARGET"

df_model = df[features + [target]].replace([np.inf, -np.inf], np.nan).dropna()

print("Model dataset shape:", df_model.shape)
df_model.head()


## 7. Split into train and test samples

This allows us to evaluate model performance on unseen data.


In [ ]:
X = df_model[features].copy()
y = df_model[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)


## 8. Train the logistic regression model

Logistic regression is widely used in credit risk because it is:

- interpretable
- stable
- easy to validate
- a natural foundation for later scorecard work


In [ ]:
model = LogisticRegression(max_iter=1000, solver="liblinear")
model.fit(X_train, y_train)


## 9. Evaluate discrimination power

We use ROC AUC as the main baseline metric.


In [ ]:
y_pred_proba = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_pred_proba)
print("Baseline AUC:", round(auc, 4))


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, linewidth=2, label=f"Baseline ROC (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, label="Random model")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Baseline Application Model")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Review model coefficients

The coefficient table helps explain which variables increase or reduce default risk.


In [ ]:
coef_table = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_[0]
})

coef_table["odds_ratio"] = np.exp(coef_table["coefficient"])
coef_table["direction"] = np.where(
    coef_table["coefficient"] > 0,
    "Higher default risk",
    "Lower default risk"
)

coef_table = coef_table.sort_values("coefficient", ascending=False).reset_index(drop=True)
coef_table


## 11. Export key outputs

These exports make the notebook easier to connect with later notebooks and portfolio review.


In [ ]:
predictions_test = pd.DataFrame({
    "actual_target": y_test.values,
    "pred_pd": y_pred_proba
})

model_metrics = pd.DataFrame({
    "metric": ["baseline_auc", "train_rows", "test_rows", "n_features"],
    "value": [auc, len(X_train), len(X_test), len(features)]
})

selected_features_df = pd.DataFrame({"feature": features})

predictions_test.to_csv(OUTPUT_DIR / "baseline_application_test_predictions.csv", index=False)
model_metrics.to_csv(OUTPUT_DIR / "baseline_application_model_metrics.csv", index=False)
selected_features_df.to_csv(OUTPUT_DIR / "baseline_application_features.csv", index=False)
coef_table.to_csv(OUTPUT_DIR / "baseline_application_coefficients.csv", index=False)

print("Baseline notebook outputs exported.")
print(sorted([p.name for p in OUTPUT_DIR.iterdir() if p.is_file()])[:20])


## 12. Portfolio wrap-up

### What this notebook demonstrates
- a clean baseline PD workflow using application data only
- simple and explainable feature engineering
- logistic regression as an interpretable benchmark
- exported outputs for later comparison

### Why this matters
This notebook creates the baseline reference point.

The next steps are to:
- engineer external bureau and behavioural features
- compare baseline vs enriched model performance
- translate the model into a scorecard workflow

That progression mirrors how a practical credit-risk portfolio project should be structured.
